In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/cicidscollection/cic-collection.parquet
/kaggle/input/bot-iot-data-set/UNSW_2018_IoT_Botnet_Full5pc_4.csv
/kaggle/input/bot-iot-data-set/UNSW_2018_IoT_Botnet_Full5pc_3.csv
/kaggle/input/bot-iot-data-set/UNSW_2018_IoT_Botnet_Full5pc_2.csv
/kaggle/input/bot-iot-data-set/UNSW_2018_IoT_Botnet_Full5pc_1.csv


In [2]:
import pandas as pd

df = pd.read_parquet("/kaggle/input/cicidscollection")

In [3]:
from sklearn.model_selection import train_test_split

In [4]:
X = df.drop('Label', axis=1)
y = df['Label']

# Perform the train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.99, random_state=42)


In [5]:


train_data = pd.concat([X_train, y_train], axis=1)



In [6]:
majority_class  = [
'    Benign     ' ,      
'DDoS-LOIC-HTTP ' ,      
'DoS-Hulk       ' ,      
'DDoS-HOIC      ' ,      
'Botnet         ' ,      
'DDoS           ' ,      
'DDoS-NTP       ' ,      
'DDoS-TFTP      ' ,      
'Bruteforce-SSH ' ,      
'Infiltration'
    
    
]
minority_class= ['DoS-Goldeneye'             
,'DDoS-Syn            '      
,'DDoS-UDP            '      
,'DoS-Slowloris       '      
,'DDoS-MSSQL          '      
,'DDoS-UDPLag         '      
,'Bruteforce-FTP      '      
,'DoS-Slowhttptest    '      
,'DDoS-Ddossim        '      
,'DDoS-DNS            '      
,'DoS-Slowread        '      
,'Portscan            '      
,'DDoS-LDAP           '      
,'Webattack-bruteforce'      
,'DDoS-SNMP           '      
,'DDoS-Slowloris      '      
,'DoS-Slowheaders     '      
,'Webattack-XSS       '      
,'DoS-Rudy            '      
,'DDoS-NetBIOS        '      
,'DoS-Slowbody        '      
,'Webattack-SQLi      '    ,  
'DoS-Heartbleed      '    ]  

# Split the dataset into minority and majority

minority = train_data[train_data['Label'].isin(minority_class)].reset_index(drop=True)
majority = train_data[train_data['Label'].isin(majority_class)].reset_index(drop=True)

In [7]:
from sklearn.neighbors import NearestNeighbors


In [8]:
majority_numeric = majority.drop(columns=['Label']).select_dtypes(include=[float, int])
minority_numeric = minority.drop(columns=['Label']).select_dtypes(include=[float, int])

# Fit NearestNeighbors
enn = NearestNeighbors(n_neighbors=500)
enn.fit(majority_numeric)

# Find difficult examples
difficult_idx = enn.kneighbors(minority_numeric, return_distance=False).flatten()

# Retrieve difficult examples
difficult = df.loc[difficult_idx]

# Drop difficult examples to find easy examples
easy = df.reset_index(drop=True).drop(difficult_idx)

print(difficult)
print(easy)

     Flow Duration  Total Fwd Packets  Total Backward Packets  \
846        5836520                  8                       7   
515      111467267                208                     192   
100       74071364                  8                       8   
233          14942                 17                      11   
851              3                  2                       0   
..             ...                ...                     ...   
884       69000206                  4                       4   
555       10177065                  7                       9   
217       69045210                  2                       2   
117       69032052                  8                       8   
903       68077536                  8                       8   

     Fwd Packets Length Total  Bwd Packets Length Total  \
846                     807.0                    3582.0   
515                   42632.0                   55246.0   
100                     384.0             

In [9]:
easy

,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Mean,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,ClassLabel
138,1159444,105,134,1217.0,291540.0,752.0,11.590476,79.662231,4344.0,2175.671631,...,0.0,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,Benign,Benign
343,3,2,0,31.0,0.0,31.0,15.500000,21.920311,0.0,0.000000,...,0.0,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,Benign,Benign
354,28,2,0,31.0,0.0,31.0,15.500000,21.920311,0.0,0.000000,...,0.0,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,Benign,Benign
366,939,7,4,2794.0,2830.0,1388.0,399.142853,675.523376,1415.0,707.500000,...,0.0,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,Benign,Benign
369,830,9,4,642.0,2944.0,306.0,71.333336,133.067657,1472.0,736.000000,...,0.0,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,Benign,Benign
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9167576,44797921,6,5,36.0,0.0,6.0,6.000000,0.000000,0.0,0.000000,...,23663.5,108.36512,23727.0,23502.0,10000000.0,72.038765,10000000.0,10000000.0,Benign,Benign
9167577,49,3,0,76.0,0.0,45.0,25.333334,23.028967,0.0,0.000000,...,0.0,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,Benign,Benign
9167578,1286687,41,42,2664.0,6954.0,456.0,64.975609,109.864571,976.0,165.571426,...,0.0,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,Benign,Benign
9167579,217,2,1,31.0,6.0,31.0,15.500000,21.920311,6.0,6.000000,...,0.0,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,Benign,Benign


In [10]:
difficult

,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Mean,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,ClassLabel
846,5836520,8,7,807.0,3582.0,437.0,100.875000,156.123978,1460.0,511.714294,...,0.000000e+00,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,Benign,Benign
515,111467267,208,192,42632.0,55246.0,3085.0,204.961533,363.816437,3254.0,287.739594,...,3.272574e+05,431549.15625,925071.0,55.0,21966184.0,10984678.0,30011433.0,9911117.0,Benign,Benign
100,74071364,8,8,384.0,384.0,48.0,48.000000,0.000000,48.0,48.000000,...,1.506835e+07,0.00000,15068346.0,15068346.0,58922768.0,0.0,58922769.0,58922769.0,Benign,Benign
233,14942,17,11,3452.0,6654.0,1313.0,203.058823,425.778473,3069.0,604.909119,...,0.000000e+00,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,Benign,Benign
851,3,2,0,37.0,0.0,31.0,18.500000,17.677670,0.0,0.000000,...,0.000000e+00,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,Benign,Benign
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
884,69000206,4,4,192.0,192.0,48.0,48.000000,0.000000,48.0,48.000000,...,1.220000e+02,0.00000,122.0,122.0,68999928.0,0.0,68999926.0,68999926.0,Benign,Benign
555,10177065,7,9,793.0,5999.0,458.0,113.285713,169.505859,1436.0,666.555542,...,1.768250e+05,0.00000,176825.0,176825.0,10000236.0,0.0,10000236.0,10000236.0,Benign,Benign
217,69045210,2,2,96.0,96.0,48.0,48.000000,0.000000,48.0,48.000000,...,4.572800e+04,0.00000,45728.0,45728.0,68954208.0,0.0,68954211.0,68954211.0,Benign,Benign
117,69032052,8,8,384.0,384.0,48.0,48.000000,0.000000,48.0,48.000000,...,1.103003e+07,0.00000,11030031.0,11030031.0,57956516.0,0.0,57956516.0,57956516.0,Benign,Benign


In [11]:
from sklearn.cluster import KMeans

In [12]:
# difficult_sample = difficult.sample(n=10000, random_state=42)  # Adjust sample size as needed

# Ensure the sample is numeric for KMeans
difficult_sample_numeric = difficult.drop(columns=['Label']).select_dtypes(include=[float, int])

# Perform KMeans clustering
kmeans = KMeans(n_clusters=100, random_state=42)
kmeans.fit(difficult_sample_numeric)

# Create compressed DataFrame
compressed = pd.DataFrame(kmeans.cluster_centers_, columns=difficult_sample_numeric.columns)
compressed['Label'] = 'Benign'



/opt/conda/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


In [13]:
compressed

,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Mean,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,1.063126e+08,5.000000,3.000000e+00,303.532995,2.470000e+02,285.532995,60.706599,1.257086e+02,2.410000e+02,8.233334e+01,...,2.000000e+01,-3.376044e-09,-1.164153e-09,-2.793968e-09,-3.143214e-09,-3.539026e-08,2.095476e-09,-3.725290e-08,-6.332994e-08,Benign
1,5.302621e+06,5.200658,3.049809e+00,164.124878,1.386930e+03,91.892200,21.482393,3.451697e+01,5.533087e+02,2.234376e+02,...,2.057316e+01,3.690366e-08,-1.234002e-08,7.823110e-08,-5.587935e-09,-3.874302e-07,6.891787e-08,-1.806766e-07,-4.116446e-07,Benign
2,6.350941e+07,291.000000,-2.131628e-14,22687.000000,-5.456968e-11,157.000000,77.962196,4.447838e+01,-2.046363e-12,5.684342e-13,...,-8.388531e+07,3.605350e+06,3.615973e+06,7.001963e+06,1.400000e+01,1.227200e+07,6.655840e+06,2.223195e+07,8.502074e+06,Benign
3,6.443768e+07,2.000000,2.000000e+00,96.000000,9.600000e+01,48.000000,48.000000,-1.023182e-12,4.800000e+01,4.800000e+01,...,2.215545e+01,6.906359e+04,4.190952e-09,6.906359e+04,6.906359e+04,6.429953e+07,-1.722947e-08,6.429953e+07,6.429953e+07,Benign
4,8.359195e+07,8.000000,8.000000e+00,384.000000,3.840000e+02,48.000000,48.000000,-2.842171e-13,4.800000e+01,4.800000e+01,...,2.090361e+01,5.065208e+06,7.070866e+06,1.006507e+07,6.535033e+04,3.669834e+07,2.741759e+07,5.608550e+07,1.731117e+07,Benign
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,1.023747e+07,10.478008,1.419728e+01,2937.561449,1.359656e+04,1471.285899,272.570658,5.178286e+02,1.894831e+03,8.070612e+02,...,2.000000e+01,3.727222e+05,1.280569e-09,3.727222e+05,3.727222e+05,9.864749e+06,-6.984919e-09,9.864749e+06,9.864749e+06,Benign
96,8.505023e+07,9.000000,9.000000e+00,432.000000,4.320000e+02,48.000000,48.000000,0.000000e+00,4.800000e+01,4.800000e+01,...,-8.388531e+07,6.050324e+06,5.656704e+06,1.005022e+07,2.050430e+06,3.644966e+07,2.757730e+07,5.594976e+07,1.694957e+07,Benign
97,3.405859e+06,37.000000,5.600000e+01,138094.000000,6.576000e+03,23360.000000,3732.270264,4.131376e+03,1.460000e+03,1.174286e+02,...,2.000000e+01,-1.513399e-09,-7.566996e-10,-1.396984e-09,-4.656613e-10,-7.450581e-09,3.492460e-09,4.097819e-08,-2.607703e-08,Benign
98,1.176822e+08,59.032206,6.959742e+01,5710.647343,7.708192e+04,499.376812,95.385692,1.707476e+02,2.628414e+03,1.028243e+03,...,3.082126e+01,4.637078e+05,1.421132e+06,4.748581e+06,3.510729e+04,1.018937e+07,1.379352e+04,1.020885e+07,1.016155e+07,Benign


In [14]:


difficult_min = difficult[difficult['Label'].isin(minority_class)]
difficult_max = difficult[difficult['Label'] == 'Benign']
difficult_range = difficult_max - difficult_min
difficult_samples = pd.DataFrame()

In [15]:
import numpy as np 

In [16]:

for i in range(10):
  r = pd.DataFrame(columns=difficult_min.columns[:-1])
  random_values = np.random.rand(difficult_min.shape[1]-1)
  r.loc[0] = random_values
  dm = difficult_min.iloc[:,:-1]
  sample = dm.add(r, fill_value=0)
  sample['Label'] = minority_class[np.random.randint(0, len(minority_class))]
  difficult_samples = pd.concat([difficult_samples, sample], ignore_index=True)
     

new_train_set = pd.concat([easy, compressed, difficult_min, difficult_samples])
# Select 10% of the rows from new_train_set
df3 = pd.concat([easy, compressed, difficult_min, difficult_samples]).sample(frac=0.8).reset_index(drop=True)


df3

,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Mean,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,ClassLabel
0,10529.0,3.0,4.0,326.0,129.0,326.0,108.666664,188.216187,112.0,32.250000,...,0.00,0.00000,0.0,0.0,0.0,0.00,0.0,0.0,Botnet,Botnet
1,475.0,1.0,1.0,48.0,96.0,48.0,48.000000,0.000000,96.0,96.000000,...,0.00,0.00000,0.0,0.0,0.0,0.00,0.0,0.0,Benign,Benign
2,1560469.0,3.0,0.0,150.0,0.0,50.0,50.000000,0.000000,0.0,0.000000,...,0.00,0.00000,0.0,0.0,0.0,0.00,0.0,0.0,Benign,Benign
3,1686720.0,8.0,7.0,1128.0,1581.0,661.0,141.000000,222.623322,1173.0,225.857147,...,0.00,0.00000,0.0,0.0,0.0,0.00,0.0,0.0,Infiltration,Infiltration
4,24532076.0,2.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,...,0.00,0.00000,0.0,0.0,24500000.0,0.00,24500000.0,24500000.0,DDoS-LOIC-HTTP,DDoS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7333375,119066734.0,17.0,16.0,883.0,1576.0,436.0,51.941177,144.555130,788.0,98.500000,...,21635.75,24084.09375,74176.0,11206.0,9899622.0,329946.25,10000000.0,8853775.0,Benign,Benign
7333376,53690241.0,2.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,...,0.00,0.00000,0.0,0.0,0.0,0.00,0.0,0.0,Benign,Benign
7333377,22315.0,1.0,1.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,...,0.00,0.00000,0.0,0.0,0.0,0.00,0.0,0.0,Benign,Benign
7333378,1585.0,5.0,2.0,935.0,299.0,935.0,187.000000,418.144714,299.0,149.500000,...,0.00,0.00000,0.0,0.0,0.0,0.00,0.0,0.0,Benign,Benign


In [17]:
# Save df3 as a CSV file
df3.to_csv('df3.csv', index=False)


In [18]:
# import numpy as np
# import pandas as pd

# def generate_samples(num_samples, difficult_min, minority_class):
#     for _ in range(num_samples):
#         random_values = np.random.rand(difficult_min.shape[1] - 1)
#         r = pd.DataFrame([random_values], columns=difficult_min.columns[:-1])
#         dm = difficult_min.iloc[:,:-1]
#         sample = dm.add(r, fill_value=0).iloc[0]
#         sample['Label'] = np.random.choice(minority_class)
#         yield sample

# # Initialize variables
# num_samples = 10000000  # Total number of samples to generate

# # Create a generator for samples
# sample_generator = generate_samples(num_samples, difficult_min, minority_class)

# # Initialize dataframes for concatenation
# difficult_samples = pd.DataFrame(columns=difficult_min.columns)
# batch_size = 10000  # Number of samples to process per batch

# # Process samples in batches and concatenate
# for batch in range(0, num_samples, batch_size):
#     batch_samples = pd.DataFrame([next(sample_generator) for _ in range(batch_size)], columns=difficult_min.columns)
#     difficult_samples = pd.concat([difficult_samples, batch_samples], ignore_index=True)
#     print(f"Processed batch {batch // batch_size + 1}")

# # Concatenate with other dataframes
# new_train_set = pd.concat([easy, compressed, difficult_min, difficult_samples])

# # Shuffle the concatenated dataframe
# df3 = new_train_set.sample(frac=1).reset_index(drop=True)

# # Save the resulting dataframe to a CSV file
# df3.to_csv('df3.csv', index=False)

# # Use df3 for further processing or analysis
# print(df3.head())


In [19]:
# import numpy as np
# import pandas as pd

# def generate_samples(num_samples, difficult_min, minority_class):
#     for _ in range(num_samples):
#         random_values = np.random.rand(difficult_min.shape[1] - 1)
#         r = pd.DataFrame([random_values], columns=difficult_min.columns[:-1])
#         dm = difficult_min.iloc[:, :-1]
#         sample = dm.add(r, fill_value=0).iloc[0]
#         sample['Label'] = np.random.choice(minority_class)
#         yield sample

# # Initialize variables
# num_samples = 10000000  # Total number of samples to generate

# # Create a generator for samples
# sample_generator = generate_samples(num_samples, difficult_min, minority_class)

# # Initialize dataframes for concatenation
# difficult_samples = []

# batch_size = 10000  # Number of samples to process per batch

# # Process samples in batches and concatenate
# for batch in range(0, num_samples, batch_size):
#     batch_samples = pd.DataFrame([next(sample_generator) for _ in range(batch_size)], columns=difficult_min.columns)
#     difficult_samples.append(batch_samples)
#     # To manage memory, we save batches to a CSV file instead of keeping all data in memory
#     batch_samples.to_csv(f'difficult_samples_batch_{batch}.csv', index=False)

# # Read all batch CSV files and concatenate into a single DataFrame
# difficult_samples = pd.concat([pd.read_csv(f'difficult_samples_batch_{batch}.csv') for batch in range(0, num_samples, batch_size)], ignore_index=True)

# # Concatenate with other dataframes
# new_train_set = pd.concat([easy, compressed, difficult_min, difficult_samples])

# # Shuffle the concatenated dataframe
# df3 = new_train_set.sample(frac=1).reset_index(drop=True)

# # Save the final DataFrame to a CSV file
# df3.to_csv('df3_data.csv', index=False)

# # Use df3 for further processing or analysis
# print(df3.head())


In [20]:
# # Save df3 to a CSV file
# df3.to_csv('df3_data.csv', index=False)
